# Roast my image

**Difficulty:** Beginner

Roast a photo two ways. A **traditional** block asks what photo judges have asked for decades: is it sharp, well exposed, well composed, well lit? A second block looks for signs that the image was **computed, generated or heavily edited**. A final cell turns the answers into tables.

This cookbook is original to this repo. Its rubrics come from the research note [`research/photo-criticism-rubrics.md`](../../research/photo-criticism-rubrics.md), which says how well each kind of judgement holds up. Only technical quality has strong human agreement. Every rubric in the second block is untested theory.

Ask your agent if anything's unclear.

In [ ]:
// Paths resolve from this notebook's folder, like the imports.
const photoBytes = await Deno.readFile("../../assets/images/IMG_1494.jpg");

// Show the exact bytes the request below sends.
Deno.jupyter.image(photoBytes);

## Add the camera's metadata

The API removes all metadata from an image before the model sees it ([Preparing images](https://docs.typesafe.ai/images#preparing-images)). So EXIF only counts if you send it yourself, as text in the `state`. The next cell reads a fixed list of fields with [`exifr`](https://github.com/MikeKovarik/exifr) and never reads GPS, so a location in your own photo stays on your machine.

Treat EXIF as a claim to check, not as proof. It's easy to strip, edit or copy from another file: a missing block doesn't mean the image is AI-generated, and a full one doesn't mean it's real. That's general image-forensics practice, not a TypeSafe claim. What it gives the model is something to cross-check against the pixels, such as the lens, aperture and whether the camera says it shot the scene directly.

In [ ]:
import exifr from "npm:exifr@7.1.3";

// A fixed list of fields. GPS is never read.
const exif = await exifr.parse(photoBytes, {
  gps: false,
  reviveValues: false, // keep dates as the camera wrote them, without a timezone shift
  pick: [
    "Make", "Model", "LensModel", "Software", "FNumber", "ExposureTime", "ISO",
    "FocalLengthIn35mmFormat", "Flash", "SceneType", "CompositeImage",
    "DateTimeOriginal", "OffsetTimeOriginal",
  ],
}) ?? {};

// Readable units, and only the fields the file actually has.
const fields: Record<string, string | number | undefined> = {
  device: exif.Make && exif.Model ? `${exif.Make} ${exif.Model}` : undefined,
  lens: exif.LensModel,
  software: exif.Software,
  aperture: exif.FNumber ? `f/${exif.FNumber}` : undefined,
  exposure: exif.ExposureTime ? `1/${Math.round(1 / exif.ExposureTime)} s` : undefined,
  iso: exif.ISO,
  focal_length_35mm: exif.FocalLengthIn35mmFormat ? `${exif.FocalLengthIn35mmFormat} mm` : undefined,
  flash: exif.Flash,
  scene_type: exif.SceneType,
  composite_image: exif.CompositeImage,
  captured: exif.DateTimeOriginal
    ? `${exif.DateTimeOriginal}${exif.OffsetTimeOriginal ? ` (UTC${exif.OffsetTimeOriginal})` : ""}`
    : undefined,
};
const camera: Record<string, string | number> = {};
for (const [name, value] of Object.entries(fields)) if (value !== undefined) camera[name] = value;

camera;

## Block 1: traditional critique

These are the criteria most often named by published judging schemes, critics and rating datasets: composition, technical quality (sharpness, exposure, noise), light, colour and subject (research note, Part 1e). Raters agree most on technical quality and least on overall merit, so impact, storytelling and originality are left out.

Every Score level describes a situation you could point to in the photo, not a degree like "okay" or "great". The [Score docs](https://docs.typesafe.ai/primitives/score) explain why: the model reads each level on its own, without its number or its neighbours.

In [ ]:
import { choice, noul, score } from "npm:@typesafe-ai/sdk@^0.7.0";

const traditional = {
  sharpness: score("How sharp is the part of `photo` that matters most?", [
    "The main subject is blurred by missed focus or camera shake.",
    "Parts of the subject are sharp, but the point that matters (eyes, the key detail) is soft.",
    "The point that matters is crisp; softness elsewhere follows depth of field.",
  ]),
  exposure: score("How is `photo` exposed?", [
    "Large areas of important detail are lost to pure black or pure white.",
    "Detail survives, but the subject is clearly too dark or too bright for the scene.",
    "The subject is well exposed; only small highlight or shadow areas clip.",
    "Highlights and shadows both keep detail across the frame.",
  ]),
  noise: score("How much noise or grain shows in `photo`?", [
    "Grain or colour speckle is visible across the subject at normal viewing size.",
    "Noise shows in shadows and flat areas, but the subject is clean.",
    "No visible noise anywhere.",
  ]),
  composition: score("How does the framing of `photo` handle its subject?", [
    "No clear subject, or the subject is awkwardly cut by the frame or lost among competing elements.",
    "A clear subject, but distracting edges, a tilted horizon or empty space pull attention away.",
    "A clear subject with clean edges, arranged as a straightforward record.",
    "Placement, edges and lines all lead to the subject; no crop would improve it.",
  ]),
  light: score("What does the light do for the subject of `photo`?", [
    "Flat or harsh light hides the subject's shape: no modelling, or hard shadows across key features.",
    "The light shows the subject clearly but adds nothing.",
    "The light's direction or quality shapes the subject and sets a mood.",
  ]),
  colour: score("How does colour work in `photo`?", [
    "Colours clash, or a cast makes familiar things (skin, sky, white objects) look wrong.",
    "Accurate, natural colour with no particular relationship between hues.",
    "A deliberate palette: hues relate to each other and support the subject.",
  ]),
  subject_emphasis: score("How easily does the eye find the subject of `photo`?", [
    "Nothing stands out; the eye has nowhere to land.",
    "The subject can be found but competes with elements of similar weight.",
    "The eye goes straight to the subject.",
  ]),
  biggest_problem: choice("What is the biggest problem with `photo`?", {
        lighting: "Too dark, too bright, or flat lighting.",
        composition: "Awkward framing or no clear subject.",
        focus: "Blurry or out of focus.",
        clutter: "Too much going on.",
        none: "Nothing stands out.",
      }),
};

## Block 2: AI-generated or heavily edited

Photo contests enforce authenticity with original files and provenance metadata, not by looking. A downscaled copy with its metadata stripped is exactly the input they refuse to rely on. People are poor at this too: about 62% correct in Microsoft's study of roughly 287,000 judgements (research note, Part 2). So this block **describes** what it sees and never delivers a verdict.

- **Gates.** `has_person` and `has_shallow_dof` are Nouls that decide whether a gated question applies. Jev answers every question on its own and has no "not applicable" level, so the table shows a gated answer as n/a when its gate is no.
- **Rubrics** from the research note: `computational_look` (rubric 1), `synthetic_bokeh_plausibility` (2; hair-level errors need a crop, which this cookbook doesn't send) and `lighting_physics_consistency` (4).
- **Cheap flags** the note recommends as Nouls: pseudo-text, extra or fused fingers, and missing contact shadows.
- **`origin_triage`** (rubric 5) is restricted. It's asked, but never shown as a row. The table turns it into one action: ask for the original file, or not.

`generative_detail_coherence` (rubric 3) is left out. It needs full-resolution crops, and `IMG_1494.jpg` is a 1280 px export.

In [ ]:
const aiOrEdited = {
  // Gates: the table shows a gated question as n/a when its gate is no.
  has_person: noul("Is there a person in `photo`?"),
  has_shallow_dof: noul("Does `photo` have a sharp subject against a noticeably blurred background?"),

  computational_look: choice("Which processing regime does the rendering of `photo` most resemble?", {
    restrained_camera: "Restrained camera rendering: shadows stay dark where the scene was dark, highlights roll off, fine texture shows grain rather than smoothing.",
    smartphone_default: "Default smartphone rendering: shadows lifted so dark areas look evenly lit, flat midtone contrast, skies held against bright foregrounds, crisp edge halos, smoothed flat areas.",
    stylised_edit: "Deliberate stylised edit: a uniform colour cast or preset applied across the frame, such as teal-orange or faded blacks.",
    synthetic_looking: "Synthetic-looking rendering: uniformly clean surfaces with no sensor noise anywhere, including dark regions.",
  }),
  synthetic_bokeh_plausibility: score("With `photo`, and the camera settings in `camera`, how consistently does background blur follow the depth of the scene?", [
        "The blur boundary cuts through the subject: hair, glasses arms, ear edges or held objects are blurred while adjacent background is sharp, or a halo of sharp background hugs the outline.",
        "Depth is mostly coherent, but one region is wrong: e.g. a gap between arm and torso keeps sharp background, or a foreground object at subject distance is blurred.",
        "Blur increases with distance and the ground plane falls off gradually; the only uncertainty is in fine hair.",
        "Foreground and background blur both scale with distance, out-of-focus highlights are consistent discs, and no outline artefacts are visible.",
      ]),
  lighting_physics_consistency: score("With `photo`, do shadows, highlights and reflections across the whole frame agree on where the light comes from?", [
        "Contradictory: shadows from objects in the same sunlight point in clearly different directions, or a reflection is missing or shows a different scene.",
        "One localised mismatch: one element (sky, person, object) is lit from a different direction or with a different colour than its surroundings, or sits without a contact shadow.",
        "Consistent overall, with an anomaly explainable by a second visible or plausible source (window, flash).",
        "Every shadow, highlight and reflection checked is consistent with the visible light sources.",
      ]),

  pseudo_text_present: noul("Does `photo` contain glyph-like shapes that look like text but don't form real letters or words?"),
  extra_or_fused_digits: noul("Does a hand in `photo` show extra, missing or fused fingers?"),
  subject_without_contact_shadow: noul("Does a person or object in `photo` rest on a surface without the contact shadow the scene's light would cast?"),
  metadata_consistent: noul("Is the camera metadata in `camera` consistent with what `photo` shows?"),

  // Restricted: used only to decide whether to ask for the original file.
  origin_triage: choice("With `photo`, and its camera metadata in `camera`, which description best fits how the image appears to have been made?", {
    single_capture: "A single camera capture with global adjustments.",
    local_compositing: "A camera capture with visible local compositing or retouching.",
    generated: "Wholly or partly generated imagery.",
    cannot_determine: "Cannot be determined from this copy.",
  }),
};

In [ ]:
import { image } from "npm:@typesafe-ai/sdk@^0.7.0";
import { client } from "../../assets/typesafe-client.ts";

const DANGER_ZONE = { headers: { "TypeSafe-Danger-Zone-Images": "true" } };

const photo = image(photoBytes);

// Both blocks in one request: extra questions cost tokens, not round trips.
const result = await client.systemOne(
  {
    state: { photo, camera: Object.keys(camera).length ? camera : "No camera metadata in the file." },
    questions: { ...traditional, ...aiOrEdited },
  },
  DANGER_ZONE,
);

console.log(JSON.stringify(result.answers, null, 2));
console.log("input tokens, image included:", result.usage.input_tokens);

## The results, as tables

The JSON above is complete but hard to scan. The next cell prints it as two tables with `console.table`, one per block:

- A **Score** shows its position (for example `1.9 / 2`, where 0 is the first level), its confidence, and the **closest level**: the full text of the level the model found most likely, with that level's probability. The position is a probability-weighted average, so when the model splits between levels it can sit near a level it barely chose. The closest level column shows what it actually picked. Low confidence means the probability was spread across levels.
- A **Choice** shows the picked option and its confidence.
- A **Noul** shows yes or no. A Noul has no confidence field ([Noul docs](https://docs.typesafe.ai/primitives/noul)), so its last column, marked `*`, is the probability of the answer shown instead.

`origin_triage` doesn't get a row. After the AI table comes one line, **ask for the original file?**, which says yes only when `origin_triage` leans towards editing or generation with high confidence **and** an independent check agrees (research note, rubric 5). The thresholds are examples: tune them on your own images.

In [ ]:
// Example thresholds. Tune them on your own images.
const GATE = 0.5; // a gate Noul at or above this counts as yes
const FLAG = 0.5; // a flag Noul at or above this counts as raised
const ORIGIN_CONFIDENCE = 0.8; // origin_triage must be at least this sure

// Gated questions, and why they don't apply when the gate is no.
const gates: Record<string, { gate: string; reason: string }> = {
  synthetic_bokeh_plausibility: { gate: "has_shallow_dof", reason: "no shallow depth of field" },
  extra_or_fused_digits: { gate: "has_person", reason: "no person" },
};

type Answer =
  | { type: "noul"; noul: number }
  | { type: "choice"; choice: string; confidence: number }
  | { type: "score"; score: number; confidence: number; legend: Record<string, string>; probabilities: Record<string, number> };
const answers = result.answers as unknown as Record<string, Answer>;

const pct = (p: number) => `${Math.round(p * 100)}%`;
const words = (id: string) => id.replaceAll("_", " ");

type Row = { question: string; answer: string; confidence: string; closest_level: string };

function row(name: string): Row {
  const question = words(name);
  const gate = gates[name];
  const gateAnswer = gate ? answers[gate.gate] : undefined;
  if (gate && gateAnswer?.type === "noul" && gateAnswer.noul < GATE) {
    return { question, answer: `n/a (${gate.reason})`, confidence: "", closest_level: "" };
  }
  const a = answers[name];
  // A Noul has no confidence field, so show the probability of the answer given, marked *.
  if (a.type === "noul") {
    return {
      question,
      answer: a.noul >= 0.5 ? "yes" : "no",
      confidence: `${pct(Math.max(a.noul, 1 - a.noul))}*`,
      closest_level: "",
    };
  }
  if (a.type === "choice") {
    return { question, answer: words(a.choice), confidence: pct(a.confidence), closest_level: "" };
  }
  // The score is a probability-weighted average, so when the probabilities split it can sit near a level
  // the model barely chose. Show the most likely level instead, with its own probability.
  const [level, probability] = Object.entries(a.probabilities).reduce((best, entry) => entry[1] > best[1] ? entry : best);
  const top = Object.keys(a.legend).length - 1;
  return {
    question,
    answer: `${a.score.toFixed(1)} / ${top}`,
    confidence: pct(a.confidence),
    closest_level: `${a.legend[level]} (${pct(probability)})`,
  };
}


// origin_triage becomes one action, and only when an independent check agrees.
const origin = answers.origin_triage;
const lighting = answers.lighting_physics_consistency;
const digits = answers.extra_or_fused_digits;
const person = answers.has_person;
const pseudoText = answers.pseudo_text_present;
const agreeing = [
  lighting.type === "score" && lighting.score <= 1 ? "the lighting check" : "",
  pseudoText.type === "noul" && pseudoText.noul >= FLAG ? "the pseudo-text flag" : "",
  person.type === "noul" && person.noul >= GATE && digits.type === "noul" && digits.noul >= FLAG ? "the fingers flag" : "",
].filter(Boolean);
const leans = origin.type === "choice" && ["local_compositing", "generated"].includes(origin.choice) &&
  origin.confidence >= ORIGIN_CONFIDENCE;
const askForOriginal = leans && agreeing.length && origin.type === "choice"
  ? `yes: the image leans towards "${words(origin.choice)}" (${pct(origin.confidence)}), and ${agreeing.join(" and ")} agree${agreeing.length === 1 ? "s" : ""}`
  : "no signal";

const traditionalRows = Object.keys(traditional).map(row);
const aiRows = Object.keys(aiOrEdited).filter((name) => name !== "origin_triage").map(row);

// Keyed by question, so each row is labelled by name rather than by number.
const byQuestion = (rows: Row[]) =>
  Object.fromEntries(rows.map(({ question, ...columns }) => [question, columns]));

console.log("Traditional critique");
console.table(byQuestion(traditionalRows));

console.log("AI-generated or heavily edited");
console.table(byQuestion(aiRows));
console.log("* A Noul has no confidence field. This is the probability of the answer shown, so 50% is a coin flip.");

console.log("Ask for the original file?", askForOriginal);